# NeuroSeg: Multi-Scale Attention U-Net with Grad-CAM++ for Brain Tumor MRI Segmentation
**Author:** Bhavik Achotani  
**Dataset:** TCGA Lower-Grade Glioma (LGG) Brain MRI Dataset (`mateuszbuda/lgg-mri-segmentation` — 3,929 scans across 110 patients)  
**Architecture:** 4-Stage Attention U-Net with Additive Soft Attention Gates ($\alpha \in [0,1]$), Custom Composite Dice-Focal Loss ($\mathcal{L} = 0.6\mathcal{L}_{\text{Dice}} + 0.4\mathcal{L}_{\text{Focal}}$), and Dense Segmentation Grad-CAM++.

In [ ]:
import glob
import os
import random
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on device:', device)

## 1. Soft Attention Gate & Attention U-Net Architecture

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, dropout_p=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        ]
        if dropout_p > 0:
            layers.insert(3, nn.Dropout2d(dropout_p))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class AttentionGate(nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1), nn.BatchNorm2d(f_int))
        self.W_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
        self.last_alpha = None

    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        alpha = self.psi(psi)
        self.last_alpha = alpha.detach()
        return x * alpha

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, use_attention=True):
        super().__init__()
        self.use_attention = use_attention
        self.pool = nn.MaxPool2d(2, 2)
        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256, 0.1)
        self.enc4 = ConvBlock(256, 512, 0.1)
        self.bottleneck = ConvBlock(512, 1024, 0.2)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.att4 = AttentionGate(512, 512, 256) if use_attention else nn.Identity()
        self.dec4 = ConvBlock(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.att3 = AttentionGate(256, 256, 128) if use_attention else nn.Identity()
        self.dec3 = ConvBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att2 = AttentionGate(128, 128, 64) if use_attention else nn.Identity()
        self.dec2 = ConvBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att1 = AttentionGate(64, 64, 32) if use_attention else nn.Identity()
        self.dec1 = ConvBlock(128, 64)

        self.head = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([self.att4(d4, e4) if self.use_attention else e4, d4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([self.att3(d3, e3) if self.use_attention else e3, d3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([self.att2(d2, e2) if self.use_attention else e2, d2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([self.att1(d1, e1) if self.use_attention else e1, d1], dim=1))
        return self.head(d1)

## 2. Composite Dice-Focal Loss & Evaluation Metrics

In [ ]:
class DiceFocalLoss(nn.Module):
    def __init__(self, dice_weight=0.6, focal_weight=0.4, alpha=0.75, gamma=2.0, smooth=1e-5):
        super().__init__()
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p_flat = probs.view(probs.size(0), -1)
        t_flat = targets.view(targets.size(0), -1).float()
        inter = (p_flat * t_flat).sum(dim=1)
        dice_loss = 1.0 - ((2.0 * inter + self.smooth) / (p_flat.sum(dim=1) + t_flat.sum(dim=1) + self.smooth)).mean()

        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = (alpha_t * ((1 - p_t) ** self.gamma) * bce).mean()
        return self.dice_weight * dice_loss + self.focal_weight * focal_loss

## 3. Run Training & Ablation Benchmark
Attach Kaggle dataset `mateuszbuda/lgg-mri-segmentation` (`/kaggle/input/lgg-mri-segmentation/kaggle_3m`) and run `!python train.py --data-dir /kaggle/input/lgg-mri-segmentation/kaggle_3m --ablation --epochs 25`.